# 03 · ET raw → Eyeball alignment

Fit `eyeball = intercept + scale * et_raw` from pupil content. Publication files are generated only in step 10.

In [1]:
import importlib
import sys
from pathlib import Path

from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the BIGPROJECT/helpers folder')

from helpers.project_config import load_project_config
from helpers.sync_quality import et_eyeball_alignment as alignment
importlib.reload(alignment)

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
PROJECT_NAME = CONFIG.project
DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
PUPIL_CSV_ROOT = CONFIG.path('paths', 'eyeball_work_root') / 'pupil_sizes'
ET_SUBFOLDER = CONFIG.value('modalities', 'eyeball_source_subfolder')
ALIGNMENT_CSV = CONFIG.path('files', 'et_eyeball_alignment_csv')
WINDOWS_CSV = CONFIG.path('files', 'et_eyeball_windows_csv')
ISSUES_CSV = CONFIG.path('files', 'et_eyeball_issues_csv')
OVERWRITE = True
PARTICIPANTS = None
VISITS = None
WINDOW_S = 90.0
STEP_S = 45.0
LOCAL_RESIDUAL_SEARCH_RADIUS_S = 15.0

In [2]:
jobs = alignment.discover_et_eyeball_jobs(
    DATA_ROOT, PUPIL_CSV_ROOT, PROJECT_NAME, et_subfolder=ET_SUBFOLDER,
    participants=PARTICIPANTS, visits=VISITS,
)
display(jobs)

,project,pid,visit,et_recording_path,pupil_csv_path,et_candidate_count,pupil_csv_candidate_count,discovery_status,et_candidates,pupil_csv_candidates
0,ImBrAid,A7G5SM,VX,E:\test\time_shifted\A7G5SM\ET\A7G5SM_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\A...,1,1,ready,E:\test\time_shifted\A7G5SM\ET\A7G5SM_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\A...
1,ImBrAid,ABVLGQ,VX,E:\test\time_shifted\ABVLGQ\ET\ABVLGQ_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\A...,1,1,ready,E:\test\time_shifted\ABVLGQ\ET\ABVLGQ_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\A...
2,ImBrAid,B9NYQ8,VX,E:\test\time_shifted\B9NYQ8\ET\B9NYQ8_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\B...,1,1,ready,E:\test\time_shifted\B9NYQ8\ET\B9NYQ8_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\B...
3,ImBrAid,CRVWF9,VX,E:\test\time_shifted\CRVWF9\ET\CRVWF9_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...,1,1,ready,E:\test\time_shifted\CRVWF9\ET\CRVWF9_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...
4,ImBrAid,CRYSEP,VX,E:\test\time_shifted\CRYSEP\ET\CRYSEP_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...,1,1,ready,E:\test\time_shifted\CRYSEP\ET\CRYSEP_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...
5,ImBrAid,CU6DM4,VX,E:\test\time_shifted\CU6DM4\ET\CU6DM4_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...,1,1,ready,E:\test\time_shifted\CU6DM4\ET\CU6DM4_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\C...
6,ImBrAid,DUJL2L,VX,E:\test\time_shifted\DUJL2L\ET\DUJL2L_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\D...,1,1,ready,E:\test\time_shifted\DUJL2L\ET\DUJL2L_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\D...
7,ImBrAid,E4MKXT,VX,E:\test\time_shifted\E4MKXT\ET\E4MKXT_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\E...,1,1,ready,E:\test\time_shifted\E4MKXT\ET\E4MKXT_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\E...
8,ImBrAid,EASFXM,VX,E:\test\time_shifted\EASFXM\ET\EASFXM_VX_et_raw,,1,0,missing_pupil_csv,E:\test\time_shifted\EASFXM\ET\EASFXM_VX_et_raw,
9,ImBrAid,FEDZAG,VX,E:\test\time_shifted\FEDZAG\ET\FEDZAG_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\F...,1,1,ready,E:\test\time_shifted\FEDZAG\ET\FEDZAG_VX_et_raw,E:\test\processed\Eyeball_Videos\pupil_sizes\F...


In [3]:
alignments, windows, issues = alignment.build_et_eyeball_alignments(
    jobs, window_s=WINDOW_S, step_s=STEP_S,
    local_residual_search_radius_s=LOCAL_RESIDUAL_SEARCH_RADIUS_S,
)
if OVERWRITE:
    alignment.save_et_eyeball_alignment_results(
        alignments, windows, issues, alignment_csv=ALIGNMENT_CSV,
        windows_csv=WINDOWS_CSV, issues_csv=ISSUES_CSV,
    )
display(alignments[[
    'pid', 'visit', 'mapping_status', 'usable', 'quality_grade',
    'review_required', 'intercept_s', 'scale', 'drift_ppm',
    'n_good_windows', 'fit_residual_p95_s', 'mapping_reason',
]])
display(issues[issues['severity'].ne('info')])

,pid,visit,mapping_status,usable,quality_grade,review_required,intercept_s,scale,drift_ppm,n_good_windows,fit_residual_p95_s,mapping_reason
0,A7G5SM,VX,content_affine,True,A,False,-1.249187,1.000016,16.295480,59.0,0.007920,local pupil-content windows
1,ABVLGQ,VX,content_affine,True,A,False,-2.600542,1.000002,2.459125,60.0,0.024313,local pupil-content windows
2,B9NYQ8,VX,content_affine,True,A,False,-2.292557,1.000014,14.369381,57.0,0.021553,local pupil-content windows
3,CRVWF9,VX,content_affine,True,A,False,-2.589389,1.000017,16.928546,55.0,0.017197,local pupil-content windows
4,CRYSEP,VX,content_affine,True,A,False,-2.122103,1.000006,6.362459,60.0,0.009737,local pupil-content windows
5,CU6DM4,VX,content_affine,True,A,False,-2.996542,1.000012,12.392064,63.0,0.005953,local pupil-content windows
6,DUJL2L,VX,content_affine,True,A,False,-2.733734,1.000006,6.023529,61.0,0.005713,local pupil-content windows
7,E4MKXT,VX,content_affine,True,A,False,-1.769248,1.000004,4.272418,60.0,0.005239,local pupil-content windows
8,EASFXM,VX,unavailable,False,D,True,NaN,NaN,NaN,NaN,NaN,missing_pupil_csv
9,FEDZAG,VX,invalid_affine,False,D,True,NaN,NaN,NaN,NaN,NaN,local lag residual P95 0.926 s exceeds hard li...


,project,pid,visit,severity,code,message
0,ImBrAid,EASFXM,VX,error,discovery_not_ready,missing_pupil_csv
1,ImBrAid,FEDZAG,VX,error,invalid_content_mapping,local lag residual P95 0.926 s exceeds hard li...
2,ImBrAid,FEDZAG,VX,warning,fit_qc,local lag residual P95 0.926 s exceeds hard limit
3,ImBrAid,FEDZAG,VX,warning,fit_qc,local lag residual P95 0.922 s exceeds hard limit
4,ImBrAid,HH5RBX,VX,warning,weak_amplitude_relation,aligned pupil amplitudes have weak global line...
5,ImBrAid,HH5RBX,VX,warning,fit_qc,fewer than 10% of candidate windows passed QC
6,ImBrAid,HH5RBX,VX,warning,fit_qc,good windows cover less than 25% of the shared...
7,ImBrAid,HKJPM3,VX,error,discovery_not_ready,missing_pupil_csv
8,ImBrAid,JVC62S,VX,warning,fit_qc,only 1 good local window(s); drift is not esti...
9,ImBrAid,N37VFK,VX,error,left_right_mapping_conflict,left/right mappings differ by 18.818 s and 0.0...
